# Module 8: Regime Detection & Adaptive Allocation

**QuantVerse** — Quantitative Portfolio Intelligence System

---

## Objectives

Markets are non-stationary — a single portfolio doesn't work in all environments.
This module detects market regimes and adapts allocation accordingly:

1. **Hidden Markov Model (HMM)** — probabilistic regime detection
2. **K-Means Clustering** — feature-based regime identification
3. **Volatility Regime** — simple percentile-based classification
4. **Adaptive Allocation** — rule-based and optimized regime-switching
5. **Walk-Forward Backtest** — adaptive vs static strategy comparison

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import warnings, logging, sys, os, json

sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO)

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12
sns.set_palette('husl')
print('Setup complete.')

In [ ]:
# Load data
data_dir = '../data/processed'
daily_returns = pd.read_parquet(f'{data_dir}/returns_daily.parquet')
clean_prices = pd.read_parquet(f'{data_dir}/prices_clean.parquet')

with open(f'{data_dir}/asset_class_map.json', 'r') as f:
    class_map = json.load(f)

signal_tickers = [t for t, c in class_map.items() if c == 'signals']
investable = [t for t in daily_returns.columns if t not in signal_tickers]
returns = daily_returns[investable].dropna()

# Portfolio weights
weights_path = f'{data_dir}/portfolio_weights.parquet'
if os.path.exists(weights_path):
    all_weights = pd.read_parquet(weights_path)
else:
    all_weights = pd.DataFrame({'Equal Weight': pd.Series(1/len(investable), index=investable)})

w_ew = pd.Series(1/len(investable), index=investable)
port_ret = pd.Series(returns.values @ w_ew.values, index=returns.index, name='EW Portfolio')

print(f'Assets: {len(investable)}, Obs: {len(returns)}')

## 1. Volatility Regime Detection (Rule-Based)

In [ ]:
from project.regime import VolatilityRegimeDetector

vol_det = VolatilityRegimeDetector(returns)
vol_regimes = vol_det.detect(port_ret, vol_window=21, lookback=252)

print('Volatility Regime Summary:')
print('=' * 60)
print(vol_det.get_regime_summary().round(4).to_string())
print(f'\nCurrent regime: {vol_det.current_regime()}')

In [ ]:
# Regime timeline
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                         gridspec_kw={'height_ratios': [2, 1]})

cum = (1 + port_ret).cumprod()
axes[0].plot(cum.index, cum.values, color='navy', lw=1.5)
axes[0].set_ylabel('Cumulative Return')
axes[0].set_title('Portfolio Equity Curve with Volatility Regimes', fontweight='bold')

# Color background by regime
regime_colors = {'Low Vol': '#4CAF50', 'Medium Vol': '#FFC107', 'High Vol': '#F44336'}
regime_s = vol_regimes['Regime']
for regime, color in regime_colors.items():
    mask = regime_s == regime
    if mask.any():
        for ax in axes:
            ax.fill_between(regime_s.index, ax.get_ylim()[0], ax.get_ylim()[1],
                           where=mask, alpha=0.15, color=color, label=regime)

axes[0].legend(fontsize=9)

axes[1].plot(vol_regimes.index, vol_regimes['Realized_Vol'] * 100, color='darkred', lw=1)
axes[1].set_ylabel('Realized Vol (%)')
axes[1].set_title('Rolling 21-Day Annualized Volatility', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Transition matrix
trans = vol_det.regime_transition_matrix()
print('Empirical Regime Transition Matrix:')
print((trans * 100).round(1).to_string())
print('(rows = from, columns = to, values = probability %)')

# Regime duration statistics (economic interpretation)
regime_s = vol_regimes['Regime']
durations = {}
current = regime_s.iloc[0]
count = 0
for r in regime_s:
    if r == current:
        count += 1
    else:
        durations.setdefault(current, []).append(count)
        current = r
        count = 1
durations.setdefault(current, []).append(count)

print('\nAverage Regime Duration (trading days):')
for regime, d in sorted(durations.items()):
    print(f'  {regime}: {np.mean(d):.0f} days (median: {np.median(d):.0f}, max: {max(d)})')

## 2. Hidden Markov Model Regime Detection

In [ ]:
from project.regime import HMMRegimeDetector

hmm = HMMRegimeDetector(returns, n_regimes=3)
hmm.fit(port_ret)

print('HMM Regime Summary:')
print('=' * 70)
print(hmm.get_regime_summary().round(4).to_string())
print(f'\nCurrent regime: {hmm.current_regime()}')
print(f'Current probabilities: {hmm.current_probabilities()}')

In [ ]:
# HMM transition matrix
hmm_trans = hmm.get_transition_matrix()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(hmm_trans * 100, annot=True, fmt='.1f', cmap='Blues',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Probability (%)'})
ax.set_title('HMM Regime Transition Probabilities', fontsize=13, fontweight='bold')
ax.set_xlabel('To')
ax.set_ylabel('From')
plt.tight_layout()
plt.show()

In [ ]:
# HMM regime probabilities over time
probs = hmm.get_regime_probabilities()

fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2]})

probs.plot.area(ax=axes[0], alpha=0.7, linewidth=0)
axes[0].set_ylabel('Probability')
axes[0].set_title('HMM Regime Probabilities Over Time', fontweight='bold')
axes[0].legend(fontsize=8, loc='upper right')

# Equity curve colored by regime
hmm_regimes = hmm.get_regime_series()
cum_aligned = cum.reindex(hmm_regimes.index)
axes[1].plot(cum_aligned.index, cum_aligned.values, color='gray', lw=0.5, alpha=0.5)

for regime, color in [('Low Vol (Bull)', 'green'), ('Medium Vol (Sideways)', 'orange'),
                       ('High Vol (Bear)', 'red')]:
    mask = hmm_regimes == regime
    if mask.any():
        axes[1].scatter(cum_aligned.index[mask], cum_aligned.values[mask],
                       c=color, s=2, alpha=0.6, label=regime)

axes[1].set_ylabel('Cumulative Return')
axes[1].set_title('Equity Curve Colored by HMM Regime', fontweight='bold')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Model selection: optimal number of regimes
# Dynamic label generation supports 2-5 regimes without label truncation
model_sel = hmm.select_n_regimes(port_ret, max_regimes=5)

fig, ax = plt.subplots(figsize=(10, 5))
model_sel['BIC'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_ylabel('BIC (lower = better)')
ax.set_title('HMM Model Selection — BIC by Number of Regimes', fontweight='bold')
ax.set_xlabel('Number of Regimes')
plt.tight_layout()
plt.show()

print(model_sel.round(1).to_string())

## 3. K-Means Clustering Regime Detection

In [ ]:
from project.regime import ClusteringRegimeDetector

clust = ClusteringRegimeDetector(returns, n_regimes=3)
clust.fit(port_ret)

print('K-Means Regime Summary:')
print('=' * 60)
print(clust.get_regime_summary().round(4).to_string())
print(f'\nCurrent regime: {clust.current_regime()}')

In [ ]:
# Elbow analysis
elbow = clust.elbow_analysis(port_ret, max_k=6)

fig, ax = plt.subplots(figsize=(8, 5))
elbow['Inertia'].plot(marker='o', ax=ax, color='steelblue', lw=2)
ax.set_ylabel('Inertia')
ax.set_title('Elbow Method for Optimal Number of Clusters', fontweight='bold')
ax.set_xlabel('K')
plt.tight_layout()
plt.show()

## 4. Regime Comparison Across Methods

In [ ]:
# Compare all three regime detection methods
vol_s = vol_regimes['Regime'].rename('Volatility')
hmm_s = hmm.get_regime_series().rename('HMM')
clust_s = clust.get_regime_series().rename('K-Means')

# Agreement rate
common_idx = vol_s.index.intersection(hmm_s.index).intersection(clust_s.index)
vol_a = vol_s.reindex(common_idx)
hmm_a = hmm_s.reindex(common_idx)
clust_a = clust_s.reindex(common_idx)

# Simple agreement: map to 0/1/2 and compare
vol_num = vol_a.map({'Low Vol': 0, 'Medium Vol': 1, 'High Vol': 2})
hmm_num = hmm_a.map({'Low Vol (Bull)': 0, 'Medium Vol (Sideways)': 1, 'High Vol (Bear)': 2})
clust_num = clust_a.map({'Low Vol': 0, 'Medium Vol': 1, 'High Vol': 2})

agree_vh = (vol_num == hmm_num).mean() * 100
agree_vc = (vol_num == clust_num).mean() * 100
agree_hc = (hmm_num == clust_num).mean() * 100

print('Regime Detection Agreement Rates:')
print(f'  Volatility vs HMM:       {agree_vh:.1f}%')
print(f'  Volatility vs K-Means:   {agree_vc:.1f}%')
print(f'  HMM vs K-Means:          {agree_hc:.1f}%')

## 5. Adaptive Allocation — Walk-Forward Backtest

In [ ]:
from project.regime import AdaptiveAllocator

allocator = AdaptiveAllocator(returns, class_map)

# Use volatility regime (most stable/interpretable)
vol_regime_series = vol_regimes['Regime']

# Rule-based adaptive (t+1 execution: today's weights use yesterday's regime signal)
adaptive_rule = allocator.adaptive_backtest(
    vol_regime_series, mode='rule_based', rebal_frequency=21
)

# Optimized adaptive
adaptive_opt = allocator.adaptive_backtest(
    vol_regime_series, mode='optimized', train_window=252, rebal_frequency=21
)

# Static benchmark (equal weight, no adaptation)
static_ret = pd.Series(returns.values @ w_ew.values, index=returns.index, name='Static EW')
static_aligned = static_ret.reindex(adaptive_rule['returns'].index)

print('Adaptive vs Static Performance:')
print('=' * 70)
for label, res in [('Adaptive (Rule)', adaptive_rule), ('Adaptive (Opt)', adaptive_opt)]:
    m = res['metrics']
    print(f"  {label:25s}  CAGR={m['CAGR']*100:.2f}%  Vol={m['Volatility']*100:.2f}%  "
          f"Sharpe={m['Sharpe']:.3f}  MaxDD={m['Max_Drawdown']*100:.2f}%  Rebal={res['n_rebalances']}")

# Static metrics
from project.backtest.metrics import PerformanceMetrics
pm_static = PerformanceMetrics(static_aligned.dropna())
print(f"  {'Static EW':25s}  CAGR={pm_static.cagr()*100:.2f}%  Vol={pm_static.annualized_volatility()*100:.2f}%  "
      f"Sharpe={pm_static.sharpe_ratio():.3f}  MaxDD={pm_static.max_drawdown()*100:.2f}%")

In [ ]:
# Equity curves: Adaptive vs Static
fig, axes = plt.subplots(2, 1, figsize=(16, 10), gridspec_kw={'height_ratios': [2, 1]}, sharex=True)

axes[0].plot(adaptive_rule['values'].index, adaptive_rule['values'].values,
             lw=1.5, label='Adaptive (Rule-Based)', color='green')
axes[0].plot(adaptive_opt['values'].index, adaptive_opt['values'].values,
             lw=1.5, label='Adaptive (Optimized)', color='blue')
static_cum = (1 + static_aligned.dropna()).cumprod()
axes[0].plot(static_cum.index, static_cum.values,
             lw=1.5, label='Static (Equal Weight)', color='gray', linestyle='--')

axes[0].set_ylabel('Portfolio Value')
axes[0].set_title('Adaptive vs Static Allocation', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)

# Regime timeline
regime_h = adaptive_rule['regime_history']
regime_colors = {'Low Vol': 'green', 'Medium Vol': 'orange', 'High Vol': 'red'}
for regime, color in regime_colors.items():
    mask = regime_h == regime
    if mask.any():
        axes[1].fill_between(regime_h.index, 0, 1, where=mask, alpha=0.4,
                            color=color, label=regime)

axes[1].set_ylabel('Regime')
axes[1].set_yticks([])
axes[1].set_title('Detected Regime Over Time', fontweight='bold')
axes[1].legend(fontsize=9, loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Asset class allocation shifts over time
wh = adaptive_rule['weights_history']
wh_df = pd.DataFrame(wh, index=adaptive_rule['returns'].index, columns=investable)

# Aggregate to asset class
ac_ts = {}
for ac in sorted(set(class_map.values()) - {'signals'}):
    tickers = [t for t, c in class_map.items() if c == ac and t in investable]
    if tickers:
        ac_ts[ac] = wh_df[tickers].sum(axis=1) * 100

ac_alloc = pd.DataFrame(ac_ts)

fig, ax = plt.subplots(figsize=(16, 7))
ac_alloc.plot.area(ax=ax, alpha=0.7, linewidth=0)
ax.set_ylabel('Allocation (%)')
ax.set_title('Adaptive Asset Class Allocation Over Time', fontsize=14, fontweight='bold')
ax.legend(fontsize=8, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Regime-Conditional Performance

In [ ]:
# How does the adaptive portfolio perform WITHIN each regime?
regime_perf = {}
for regime in ['Low Vol', 'Medium Vol', 'High Vol']:
    mask = adaptive_rule['regime_history'] == regime
    if mask.sum() > 10:
        r_adapt = adaptive_rule['returns'][mask]
        r_static = static_aligned.reindex(r_adapt.index).dropna()
        r_adapt = r_adapt.reindex(r_static.index)

        regime_perf[regime] = {
            'Adaptive_Ann_Ret_%': r_adapt.mean() * 252 * 100,
            'Static_Ann_Ret_%': r_static.mean() * 252 * 100,
            'Adaptive_Vol_%': r_adapt.std() * np.sqrt(252) * 100,
            'Static_Vol_%': r_static.std() * np.sqrt(252) * 100,
            'Adaptive_Sharpe': r_adapt.mean() / r_adapt.std() * np.sqrt(252) if r_adapt.std() > 0 else 0,
            'Static_Sharpe': r_static.mean() / r_static.std() * np.sqrt(252) if r_static.std() > 0 else 0,
            'N_Days': mask.sum(),
        }

regime_perf_df = pd.DataFrame(regime_perf).T
print('Regime-Conditional Performance: Adaptive vs Static')
print('=' * 90)
print(regime_perf_df.round(3).to_string())

## 7. Export Regime Data

In [ ]:
# Export regime series
regime_export = pd.DataFrame({
    'Vol_Regime': vol_regimes['Regime'],
    'HMM_Regime': hmm.get_regime_series(),
    'Cluster_Regime': clust.get_regime_series(),
})
regime_export.to_parquet(f'{data_dir}/regime_labels.parquet')

# Export adaptive returns
adaptive_returns = pd.DataFrame({
    'Adaptive_Rule': adaptive_rule['returns'],
    'Adaptive_Opt': adaptive_opt['returns'],
})
adaptive_returns.to_parquet(f'{data_dir}/adaptive_returns.parquet')
print('Regime labels and adaptive returns exported.')

---

## Key Takeaways from Module 8

1. **Three regime detection methods** provide complementary views — HMM is probabilistic, K-Means is feature-rich, Volatility is interpretable
2. **Markets spend most time** in Medium Vol regime; High Vol is rare but destructive
3. **Transition probabilities** show regimes are persistent (sticky) — useful for prediction
4. **Adaptive allocation improves risk-adjusted returns** primarily by reducing drawdowns in High Vol regimes
5. **Rule-based adaptation** is simpler and more robust than full re-optimization per regime
6. **The key benefit is defensive** — cutting crypto/equity exposure during crises preserves capital

---

**QuantVerse Modules 1–8 Complete.**